STEP 1

In [1]:
import pandas as pd

dirty_data = pd.read_csv('super_dirty_students.csv')

dirty_data.head(10)

dirty_data.isnull().sum()

student_id        0
name            335
age             108
gender          219
score           246
phone           371
city              0
email           151
date_of_join      0
course            0
attendance      326
status            0
gpa             141
remarks         204
money_spent       0
event_time        0
address_raw       0
profile_json      0
dtype: int64

STEP 2

In [2]:
columns_to_strip = ['name', 'gender', 'city', 'email', 'course', 'status', 'remarks']

for col in columns_to_strip:
    if col in dirty_data.columns:
        dirty_data[col] = dirty_data[col].str.strip()
        dirty_data[col] = dirty_data[col].fillna('Null')

STEP 3

In [3]:
import pandas as pd
import re
from word2number import w2n

def convert_str_int(value):
    if pd.isna(value):
        return None
    
    value = str(value).replace(',', '.')    
    
    nums = re.findall(r'-?\d+', str(value))

    if nums:
        num = int(nums[0])
        return max(num, 0)
    

def convert_str(value):
    if pd.isna(value):
        return None
    
    value = str(value).replace(',', '.')
    
    nums = re.findall(r'-?\d+\.?\d*', str(value))

    if nums:
        num = float(nums[0])
        return max(num, 0)
    
    try:
        num = float(w2n.word_to_num(str(value)))
        return max(num, 0)
    
    except ValueError:
        return None
    
dirty_data['age'] = dirty_data['age'].apply(convert_str).fillna(0).astype(int)

dirty_data['score'] = dirty_data['score'].apply(convert_str).fillna(0).astype(int)

dirty_data['gpa'] = dirty_data['gpa'].apply(convert_str).fillna(0)

dirty_data['attendance'] = dirty_data['attendance'].apply(convert_str_int).fillna(0).astype(int)

dirty_data['money_spent'] = dirty_data['money_spent'].apply(convert_str).fillna(0).astype(int)

dirty_data

,student_id,name,age,gender,score,phone,city,email,date_of_join,course,attendance,status,gpa,remarks,money_spent,event_time,address_raw,profile_json
0,1,Claudia Short,20,Null,0,+1-619-379-4152x102,Katieland,someonegmail.com,1662247364,Data Science,0,active,3.72,good,135,1629312830,"Apartment 37, South Kevin district, Tashkent, ...","{'hobbies': ['gun', 'nice'], 'skills': {'tech'..."
1,2,Null,20,Female,90,NaN,Dawnburgh,psmith@chen.com,2017/08/29,Data Science,0,active,1.88,excellent,152,11/10/2001 04:19 AM,UZ 100332 Tashkent South Patricia,"{hobbies:['against', 'good']}"
2,3,Kathryn Moyer,20,Null,90,NaN,Lake Stevenmouth,Null,2017-08-14,DATA SCIENCE,0,pending,0.00,excellent,185,1657837622,"Wendyshire 12-kv, dom 1, Tashkent","{'hobbies': ['fast', 'clearly'], 'skills': {'t..."
3,4,Ruben Wilson,20,fmale,81,NaN,Port Pamelafort,special,1973-09-17,data-sciens,0,inactive,0.00,good,175,1682795130,"Apartment 16, North Tamara district, Tashkent,...","{'hobbies': ['left', 'role'], 'skills': {'tech..."
4,5,Robert Pruitt,20,Female,0,001-182-659-5631x02803,Kingburgh,six,2023/11/25,data_sciense,0,pending,4.37,average,282,2013-10-13,"BROKEN,ADDRESS,DATA,,,","{'hobbies': ['sort', 'science'], 'skills': {'t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,Gary Sawyer,22,fmale,0,NaN,Gabriellefurt,amy97@woodard.net,1992.11.13,DATA SCIENCE,0,inactive,0.00,excellent,64,14/10/2007 06:53 PM,"Apartment 12, Laurenhaven district, Tashkent, ...","{'hobbies': ['collection', 'his'], 'skills': {..."
996,997,Scott Clark,20,Null,0,001-205-065-8737,Rogerborough,katiepage@hensley.com,2024.10.05,DATA SCIENCE,0,pending,3.70,Null,143,2021-06-21T07:46:27,"Patrickmouth 4-kv, dom 6, Tashkent",INVALID_JSON_DATA
997,998,Null,20,MALE,0,264-577-8585,South Mary,someonegmail.com,19/11/2021 02:23 AM,data_sciense,95,inactive,0.00,excellent,79,1975/05/11,"Apartment 38, New Brendan district, Tashkent, ...","{'hobbies': ['street', 'read'], 'skills': {'te..."
998,999,Brittany Barrett,21,FEMALE,86,001-663-118-1327x207,Port Rachael,richardgreen@@shannon-jenkins.info,1625723463,data_sciense,110,active,3.06,average,238,2015.05.24,"25998 Martinez Grove Apt. 473, West Scott","{'hobbies': ['game', 'total'], 'skills': {'tec..."


STEP 4

In [4]:
import pandas as pd
import re

def clean_email(evalue):
    if pd.isna(evalue):
        return None
    
    evalue = str(evalue).strip()
    evalue = re.sub(r'^[^a-zA-Z0-9]+', '', evalue)

    parts = evalue.split('@')
    if len(parts) > 1:
        evalue = parts[0] + '@' + '@'.join(parts[1:]).replace('@', '')

    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    if re.match(pattern, evalue):
        return evalue
    else:
        return None
    
def clean_phone_uz(pvalue):
    if pd.isna(pvalue):
        return None
    
    pvalue = str(pvalue).strip()

    if pvalue == '' or pvalue == 'nan':
        return None
    
    pvalue = re.sub(r'x\d+', '', pvalue, flags=re.IGNORECASE)

    digits = re.sub(r'\D', '', pvalue)

    if not digits:
        return None
    
    if digits.startswith('001'):
        digits = digits[3:]
    elif digits.startswith('1') and len(digits) == 11:
        digits[1:]

    digits = digits[-9:]

    if len(digits) < 9:
        return None
    
    return f'+998 {digits[0:2]} {digits[2:5]} {digits[5:7]} {digits[7:9]}'

dirty_data['email'] = dirty_data['email'].apply(clean_email).fillna('Null')

dirty_data['phone'] = dirty_data['phone'].apply(clean_phone_uz).fillna('Null')

dirty_data

,student_id,name,age,gender,score,phone,city,email,date_of_join,course,attendance,status,gpa,remarks,money_spent,event_time,address_raw,profile_json
0,1,Claudia Short,20,Null,0,+998 19 379 41 52,Katieland,Null,1662247364,Data Science,0,active,3.72,good,135,1629312830,"Apartment 37, South Kevin district, Tashkent, ...","{'hobbies': ['gun', 'nice'], 'skills': {'tech'..."
1,2,Null,20,Female,90,Null,Dawnburgh,psmith@chen.com,2017/08/29,Data Science,0,active,1.88,excellent,152,11/10/2001 04:19 AM,UZ 100332 Tashkent South Patricia,"{hobbies:['against', 'good']}"
2,3,Kathryn Moyer,20,Null,90,Null,Lake Stevenmouth,Null,2017-08-14,DATA SCIENCE,0,pending,0.00,excellent,185,1657837622,"Wendyshire 12-kv, dom 1, Tashkent","{'hobbies': ['fast', 'clearly'], 'skills': {'t..."
3,4,Ruben Wilson,20,fmale,81,Null,Port Pamelafort,Null,1973-09-17,data-sciens,0,inactive,0.00,good,175,1682795130,"Apartment 16, North Tamara district, Tashkent,...","{'hobbies': ['left', 'role'], 'skills': {'tech..."
4,5,Robert Pruitt,20,Female,0,+998 82 659 56 31,Kingburgh,Null,2023/11/25,data_sciense,0,pending,4.37,average,282,2013-10-13,"BROKEN,ADDRESS,DATA,,,","{'hobbies': ['sort', 'science'], 'skills': {'t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,Gary Sawyer,22,fmale,0,Null,Gabriellefurt,amy97@woodard.net,1992.11.13,DATA SCIENCE,0,inactive,0.00,excellent,64,14/10/2007 06:53 PM,"Apartment 12, Laurenhaven district, Tashkent, ...","{'hobbies': ['collection', 'his'], 'skills': {..."
996,997,Scott Clark,20,Null,0,+998 05 065 87 37,Rogerborough,katiepage@hensley.com,2024.10.05,DATA SCIENCE,0,pending,3.70,Null,143,2021-06-21T07:46:27,"Patrickmouth 4-kv, dom 6, Tashkent",INVALID_JSON_DATA
997,998,Null,20,MALE,0,+998 64 577 85 85,South Mary,Null,19/11/2021 02:23 AM,data_sciense,95,inactive,0.00,excellent,79,1975/05/11,"Apartment 38, New Brendan district, Tashkent, ...","{'hobbies': ['street', 'read'], 'skills': {'te..."
998,999,Brittany Barrett,21,FEMALE,86,+998 63 118 13 27,Port Rachael,richardgreen@shannon-jenkins.info,1625723463,data_sciense,110,active,3.06,average,238,2015.05.24,"25998 Martinez Grove Apt. 473, West Scott","{'hobbies': ['game', 'total'], 'skills': {'tec..."


STEP 5

In [5]:
import pandas as pd
import ast
import json
import re

def parse_profile(profile):
    if pd.isna(profile):
        return None
    try:
        return ast.literal_eval(str(profile))
    except:
        try:
            fixed = re.sub(r'(\w+):', r'"\1":', str(profile))
            fixed = fixed.replace("'", '"')
            return json.loads(fixed)
        except:
            return None
        
# dirty_data['hobbies'] = dirty_data['profile_json'].apply(parse_profile).apply(lambda x: x.get('hobbies') if x else None)

parsed = dirty_data['profile_json'].apply(parse_profile)

dirty_data['hobbies'] = parsed.apply(lambda x: ', '.join(x.get('hobbies', [])) if x else None)

dirty_data['skill_python'] = parsed.apply(lambda x: x.get('skills', {}).get('tech', {}).get('python') if x else None)
dirty_data['skill_excel'] = parsed.apply(lambda x: x.get('skills', {}).get('tech', {}).get('excel') if x else None)
dirty_data['skill_sql'] = parsed.apply(lambda x: x.get('skills', {}).get('tech', {}).get('sql') if x else None)

dirty_data['skill_soft'] = parsed.apply(lambda x: ', '.join(x.get('skills', {}).get('soft', [])) if x else None)

dirty_data['family_siblings'] = parsed.apply(lambda x: x.get('family', {}).get('siblings') if x else None)

dirty_data['family_income_father'] = parsed.apply(lambda x: x.get('family', {}).get('income', {}).get('father') if x else None)
dirty_data['family_income_mother'] = parsed.apply(lambda x: x.get('family', {}).get('income', {}).get('mother') if x else None)

dirty_data['device1_type1_laptop'] = parsed.apply(lambda x: next((i['type'] for i in x.get('devices', []) if i['type'] == 'laptop'), None) if x else None)
dirty_data['device1_brand1_laptop'] = parsed.apply(lambda x: next((i['brand'] for i in x.get('devices', []) if i['type'] == 'laptop'), None) if x else None)
dirty_data['device1_year1_laptop'] = parsed.apply(lambda x: next((i['year'] for i in x.get('devices', []) if i ['type'] == 'laptop'), None) if x else None)

dirty_data['device2_type2_phone'] = parsed.apply(lambda x: next((i['type'] for i in x.get('devices', []) if i['type'] == 'phone'), None) if x else None)
dirty_data['device2_brand2_phone'] = parsed.apply(lambda x: next((i['brand'] for i in x.get('devices', []) if i['type'] == 'phone'), None) if x else None)
dirty_data['device2_year2_phone'] = parsed.apply(lambda x: next((i['year'] for i in x.get('devices', []) if i['type'] == 'phone'), None) if x else None)

dirty_data.drop(columns=['profile_json'], inplace=True)

STEP 6

In [6]:
import pandas as pd
import re

def parse_address_raw(address):
    if pd.isna(address):
        return None, None, None
    
    address = str(address).strip()

    if 'BROKEN' in address.upper():
        return None, None, None

    postal = re.findall(r'\d{5,6}', address)
    postal = postal[0] if postal else None

    if address.startswith('UZ'):
        parts = address.split()
        city = parts[2] if len(parts) > 2 else None
        district = ' '.join(parts[3:]) if len(parts) > 3 else None
        return city, district, postal
    
    parts = [p.strip() for p in address.split(',')]

    city = None
    district = None
    for i, part in enumerate(parts):
        if part == 'UZ':
            city = parts[i-1] if i > 0 else None
            break
        if part == parts[-1] and not re.match(r'^\d+$', part):
            city = part

    for part in parts:
        if 'district' in part.lower():
            district = part.strip()
            break
    
    return city, district, postal

dirty_data[['addr_city', 'addr_district', 'addr_postal']] = dirty_data['address_raw'].apply(lambda x: pd.Series(parse_address_raw(x)))

STEP 7

In [7]:
print(dirty_data.duplicated().sum())

0


STEP 8

In [8]:
from difflib import get_close_matches

def data_normalization_gender(genders):
    if pd.isna(genders):
        return None
    
    genders = genders.lower()

    valid1 = ['male', 'female']

    match1 = get_close_matches(genders, valid1, n=1, cutoff=0.6)

    return match1[0].capitalize() if match1 else 'Unknown'

def data_normalization_course(courses):
    if pd.isna(courses):
        return None
    
    courses = courses.lower()

    valid2 = ['data science', 'python']

    match2 = get_close_matches(courses, valid2, n=1, cutoff=0.6)

    return match2[0].title() if match2 else 'Null'

def data_normalization_status(statuses):
    if pd.isna(statuses):
        return None
    
    statuses = statuses.capitalize()

    return statuses

dirty_data['gender'] = dirty_data['gender'].apply(data_normalization_gender)

dirty_data['course'] = dirty_data['course'].apply(data_normalization_course)

dirty_data['status'] = dirty_data['status'].apply(data_normalization_status)

In [9]:
print(dirty_data['gender'].value_counts(dropna=False))

gender
Female     501
Male       280
Unknown    219
Name: count, dtype: int64


In [10]:
print(dirty_data['course'].value_counts(dropna=False))

course
Data Science    500
Python          306
Null            194
Name: count, dtype: int64


In [11]:
print(dirty_data['status'].value_counts(dropna=False))

status
Inactive    362
Active      329
Pending     309
Name: count, dtype: int64


STEP 9

In [14]:
import pandas as pd

def parse_datetime(dvalue):
    if pd.isna(dvalue):
        return None
    
    dvalue = str(dvalue).strip()

    if dvalue.isdigit():
        return pd.to_datetime(int(dvalue), unit='s').strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        return pd.to_datetime(dvalue, dayfirst=True).strftime('%Y-%m-%d %H:%M:%S')
    except:
        return None
    
dirty_data['date_of_join'] = pd.to_datetime(dirty_data['date_of_join'].apply(parse_datetime))
dirty_data['event_time'] = pd.to_datetime(dirty_data['event_time'].apply(parse_datetime))

C:\Users\Hp\AppData\Local\Temp\ipykernel_3488\2161175134.py:13: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(dvalue, dayfirst=True).strftime('%Y-%m-%d %H:%M:%S')
C:\Users\Hp\AppData\Local\Temp\ipykernel_3488\2161175134.py:13: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(dvalue, dayfirst=True).strftime('%Y-%m-%d %H:%M:%S')


STEP 10

In [15]:
dirty_data.to_csv('cleaned_students_datas.csv')